# Output Parsers

In [6]:
from langchain.chat_models import init_chat_model
import os
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

from dotenv import load_dotenv

load_dotenv()
llm = init_chat_model("openai/gpt-oss-120b", 
                      model_provider="openai",
                      base_url="https://api.groq.com/openai/v1",
                      api_key = os.getenv("GROQ_API_KEY"))

## String Parser

In [10]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="You are a helpful assistant. Answer this question: {question}?"
)

from langchain_core.output_parsers import StrOutputParser
# Clean string output
string_parser = StrOutputParser()

# In a chain
chain = prompt | llm | string_parser
result = chain.invoke({"question": "What is Python?"})
print("Result :", result)


Result : **Python** is a high‑level, interpreted programming language that is designed to be easy to read and write. It was created by Guido van Rossum and first released in 1991. Here are some key characteristics that define Python:

| Feature | Description |
|---------|-------------|
| **Readability** | Uses indentation (whitespace) to define code blocks, which encourages clean, readable code. |
| **Interpreted** | Code is executed line‑by‑line by the Python interpreter, so you don’t need a separate compilation step. |
| **Dynamically typed** | Variable types are determined at runtime, allowing you to write flexible code without explicit type declarations. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, and functional programming styles. |
| **Extensive standard library** | Comes with batteries‑included modules for tasks like file I/O, networking, web services, data serialization, and more. |
| **Cross‑platform** | Runs on Windows, macOS, Linux, and many other operatin

## List Parser

In [11]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
# List parser
#creating the instance of the class CommaSeparatedListOutputParser
list_parser = CommaSeparatedListOutputParser()

In [12]:
list_parser.get_format_instructions()

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [33]:
prompt = PromptTemplate(
    template="give top 5 programming languages:\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": list_parser.get_format_instructions()}
)

In [34]:
chain = prompt | llm | list_parser

In [35]:
result = chain.invoke({})

In [36]:
result

['Python', 'JavaScript', 'Java', 'C#', 'C++']

In [37]:
result[0]

'Python'

## JSON Parser

In [38]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()

In [39]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [40]:
prompt = ChatPromptTemplate.from_template(
    """Analyze this product and return a JSON object with these fields:
    - name: product name
    - price: estimated price in USD
    - category: product category
    - features: array of key features
    
    Product: {product}
    
    {format_instructions}"""
).partial(format_instructions=json_parser.get_format_instructions())

In [52]:
json_chain = prompt | llm | json_parser
result = json_chain.invoke({"product": "iPhone 15 Pro"})

In [54]:
result

{'name': 'iPhone 15 Pro',
 'price': 999,
 'category': 'Smartphone',
 'features': ['A17 Bionic chip with 6‑core CPU and 5‑core GPU',
  '6.1‑inch Super Retina XDR display with ProMotion (up to 120\u202fHz refresh rate)',
  'Titanium frame for increased durability and lighter weight',
  'Triple‑camera system: 48\u202fMP main sensor, 12\u202fMP ultra‑wide, 12\u202fMP telephoto with 3x optical zoom',
  'Advanced Photonic Engine for improved low‑light photography',
  'USB‑C connector with fast charging and data transfer',
  'iOS 17 with enhanced privacy features and dynamic island UI',
  '5G connectivity, Wi‑Fi\u202f6E, and MagSafe accessories support',
  'Ceramic Shield front cover for increased drop resistance',
  'Improved battery life with up to 22\u202fhours of video playback']}

In [55]:
result["price"]

999

In [56]:
result["category"]

'Smartphone'

## Pydantic Parser

In [1]:
from datetime import datetime
from typing import Optional
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str 
    signup_ts: Optional[datetime] = None
    friends: list[int] = []



In [8]:
external_data = {'id': '123','name':'Abhishek', 'signup_ts': '2017-06-01 12:22', 'friends': [1, '2', b'3']}


In [9]:
user = User(**external_data)
print(user)

id=123 name='Abhishek' signup_ts=datetime.datetime(2017, 6, 1, 12, 22) friends=[1, 2, 3]


In [80]:
user.id

'123'

## Pydantic --LLM

In [81]:
from typing import List
from pydantic import BaseModel, Field, field_validator


from langchain_core.output_parsers import PydanticOutputParser


In [106]:
class Person(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(default_factory=int, description="Age in years")
    hobbies: List[str] = Field(default_factory=list, description="List of hobbies")

    @field_validator("age")
    def age_must_be_positive(cls, v):
        if v < 0:
            raise ValueError("age must be non-negative")
        return v

class People(BaseModel):
    people: List[Person]

In [107]:
parser = PydanticOutputParser(pydantic_object=People)

In [108]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Person": {"properties": {"name": {"description": "The person's name", "title": "Name", "type": "string"}, "age": {"description": "Age in years", "title": "Age", "type": "integer"}, "hobbies": {"description": "List of hobbies", "items": {"type": "string"}, "title": "Hobbies", "type": "array"}}, "required": ["name"], "title": "Person", "type": "object"}}, "properties": {"people": {"items": {"$ref": "#/$defs/Person"}, "title": "People", "type": "array"}}, "required": ["people"]}
```


In [109]:
prompt = PromptTemplate(
    template="""
You are given a text describing some people. Extract list of people details in JSON.
Output must follow this JSON schema exactly:
{format_instructions}

Text:
{text}
""",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)


In [116]:
input_text = "Alice is 30 years old and loves painting. Bob is 25 and enjoys cycling and chess."


In [117]:
_prompt = prompt.format_prompt(text=input_text)


chain = prompt | llm | parser
result = chain.invoke({"text": input_text})
print("Result :")
print(result)

Result :
people=[Person(name='Alice', age=30, hobbies=['painting']), Person(name='Bob', age=25, hobbies=['cycling', 'chess'])]


In [118]:
print(result.people)
print("Name :", result.people[0].name)
print("Age :", result.people[0].age)
print("Hobbies :", result.people[0].hobbies)

print("--------------------------------")


[Person(name='Alice', age=30, hobbies=['painting']), Person(name='Bob', age=25, hobbies=['cycling', 'chess'])]
Name : Alice
Age : 30
Hobbies : ['painting']
--------------------------------


In [120]:
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import PydanticOutputParser

# Define nested models for recipe
class Ingredient(BaseModel):
    name: str = Field(..., description="Ingredient name")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")
    optional: bool = Field(default=False, description="Whether ingredient is optional")

class RecipeStep(BaseModel):
    step_number: int = Field(..., description="Step number in sequence")
    instruction: str = Field(..., description="Detailed instruction for this step")
    duration_minutes: Optional[int] = Field(None, description="Time required in minutes")

class Recipe(BaseModel):
    name: str = Field(..., description="Recipe name")
    cuisine: str = Field(..., description="Type of cuisine")
    servings: int = Field(..., description="Number of servings")
    prep_time_minutes: int = Field(..., description="Preparation time in minutes")
    cook_time_minutes: int = Field(..., description="Cooking time in minutes")
    ingredients: List[Ingredient] = Field(..., description="List of ingredients")
    steps: List[RecipeStep] = Field(..., description="Cooking steps in order")
    difficulty: str = Field(..., description="Difficulty level: Easy, Medium, or Hard")
    
    @field_validator("servings")
    def servings_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("servings must be positive")
        return v
    
    @field_validator("difficulty")
    def difficulty_must_be_valid(cls, v):
        valid_levels = ["Easy", "Medium", "Hard"]
        if v not in valid_levels:
            raise ValueError(f"difficulty must be one of {valid_levels}")
        return v

# Create parser
recipe_parser = PydanticOutputParser(pydantic_object=Recipe)

# Create prompt
recipe_prompt = PromptTemplate(
    template="""
Give me a complete recipe for {recipe_topic}, and format it as JSON.
Output must follow this JSON schema exactly:
{format_instructions}
""",
    input_variables=["recipe_topic"],
    partial_variables={"format_instructions": recipe_parser.get_format_instructions()}
)

# Example usage
recipe_topic = "Chocolate Chip Cookies"

recipe_chain = recipe_prompt | llm | recipe_parser
recipe_result = recipe_chain.invoke({"recipe_topic": recipe_topic})
print(recipe_result)
print("Recipe Name:", recipe_result.name)
print("Cuisine:", recipe_result.cuisine)
print("Servings:", recipe_result.servings)
print("Total Time:", recipe_result.prep_time_minutes + recipe_result.cook_time_minutes, "minutes")
print("\nIngredients:")
for ing in recipe_result.ingredients:
    optional_text = " (optional)" if ing.optional else ""
    print(f"  - {ing.quantity} {ing.name}{optional_text}")
print("\nFirst Step:", recipe_result.steps[0].instruction)
print("Difficulty:", recipe_result.difficulty)


name='Chocolate Chip Cookies' cuisine='American' servings=24 prep_time_minutes=15 cook_time_minutes=11 ingredients=[Ingredient(name='All-purpose flour', quantity='2 1/4 cups', optional=False), Ingredient(name='Baking soda', quantity='1 teaspoon', optional=False), Ingredient(name='Salt', quantity='1 teaspoon', optional=False), Ingredient(name='Unsalted butter, softened', quantity='1 cup (2 sticks)', optional=False), Ingredient(name='Granulated sugar', quantity='3/4 cup', optional=False), Ingredient(name='Brown sugar, packed', quantity='3/4 cup', optional=False), Ingredient(name='Vanilla extract', quantity='1 teaspoon', optional=False), Ingredient(name='Large eggs', quantity='2', optional=False), Ingredient(name='Chocolate chips', quantity='2 cups', optional=False), Ingredient(name='Chopped nuts (optional)', quantity='1 cup', optional=True)] steps=[RecipeStep(step_number=1, instruction='Preheat the oven to 375°F (190°C).', duration_minutes=None), RecipeStep(step_number=2, instruction='In